# Exercise 01 — Risk and Return: The Historical Record

**MSc Finance · Investments · FHNW**

In this exercise you work with the long-run historical record of **total-return indices**
(1927–2025) for several US asset classes, compiled by A. Damodaran (NYU Stern).
You will:

1. Import the data directly from the course GitHub repository — *no local installation needed*.
2. **Compute annual returns yourself** from the index levels.
3. Visualise the **distribution** of returns.
4. Compute **descriptive statistics** (arithmetic vs. geometric mean, volatility, skewness, kurtosis, Sharpe ratio).
5. **Export** your figures and tables.

Run each cell from top to bottom (`Shift+Enter`). Cells marked **🛠 Your turn** ask you to write code.


## 1 · Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# FHNW colour palette (consistent with lecture dashboards)
FHNW = {
    "navy":   "#1B3A5C",
    "blue":   "#0057A4",
    "green":  "#2E7D32",
    "red":    "#C8102E",
    "orange": "#EA8700",
    "yellow": "#FFD500",
}

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

## 2 · Import the data

The data lives in the course repository. We read the Excel file straight from its raw URL,
so there is nothing to download by hand. We tidy up the column names and use the year as the index.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/KroeTiA/Investments/main/Exercise_01/data/histretSP_investments.xls"
SHEET = "Total Return Index_Damadoran"

raw = pd.read_excel(DATA_URL, sheet_name=SHEET)

# Keep only the data columns (drop the trailing source-note column)
raw = raw.loc[:, ~raw.columns.str.contains("Source", case=False, na=False)]
raw.columns = [c.strip() for c in raw.columns]          # trim stray spaces
idx = raw.set_index("Year")

print(f"Loaded {idx.shape[0]} annual observations, {idx.shape[1]} asset classes")
print("Assets:", list(idx.columns))
idx.head()

### Total-return indices over time

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
for col in idx.columns:
    ax.plot(idx.index, idx[col], label=col, lw=1.6)
ax.set_yscale("log")  # log scale: equal vertical distance = equal % change
ax.set_ylabel("Index level (log scale, base = 100 in 1927)")
ax.set_xlabel("Year")
ax.set_title("Historical Total-Return Indices, 1927–2025")
ax.legend(frameon=False, fontsize=8, ncol=2)
fig.tight_layout()
plt.show()

## 3 · From indices to returns  🛠 *Your turn*

A total-return **index** $I_t$ already reinvests all income (dividends, coupons). The annual
**total return** is the relative change in the index from one year to the next:

$$ R_t = \frac{I_t}{I_{t-1}} - 1 $$

**Task:** Compute a DataFrame `rets` of annual returns from `idx`.

*Hint:* pandas has a one-line method for exactly this relative change. Look up
`DataFrame.pct_change()`. Afterwards, the first row will be `NaN` (there is no prior year) —
drop it with `.dropna()`.

In [ ]:
# 🛠 Your turn: compute annual returns and store them in `rets`
# rets = ...

# --- check your result (uncomment once you have `rets`) ---
# print(rets.shape)      # expect (98, 7)
# rets.head()

<details>
<summary>💡 Click for the solution</summary>

```python
rets = idx.pct_change().dropna()
rets.head()
```
</details>

## 4 · Distribution of returns

Once `rets` exists, this cell plots a histogram for each asset class. Notice how the **spread**
differs dramatically between, say, T-Bills and Small Cap stocks.

In [ ]:
n = idx.shape[1]
ncols = 4
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3.0 * nrows))
axes = axes.flatten()
for ax, col in zip(axes, rets.columns):
    ax.hist(rets[col], bins=25, color=FHNW["blue"], alpha=0.85, edgecolor="white")
    ax.axvline(rets[col].mean(), color=FHNW["red"], ls="--", lw=1.3)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("Annual return")
for ax in axes[n:]:
    ax.set_visible(False)
fig.suptitle("Distribution of Annual Returns (dashed line = mean)", y=1.02)
fig.tight_layout()
plt.show()

## 5 · Descriptive statistics

We summarise each series. The data are already annual, so no annualisation is needed.
Watch the gap between the **arithmetic** and the **geometric** mean — it widens with
volatility, a central theme of this lecture.

The Sharpe ratio uses the **3-month T-Bill** as the risk-free asset.

In [ ]:
from scipy import stats

RF_COL = "3-month T.Bill"   # risk-free proxy

def describe(r, rf_col=RF_COL):
    rf_geo = (1 + r[rf_col]).prod() ** (1 / len(r)) - 1
    out = {}
    for col in r.columns:
        x = r[col]
        geo = (1 + x).prod() ** (1 / len(x)) - 1
        vol = x.std(ddof=1)
        out[col] = {
            "Arith. mean": x.mean(),
            "Geo. mean":   geo,
            "Volatility":  vol,
            "Skewness":    stats.skew(x),
            "Excess kurtosis": stats.kurtosis(x),
            "Sharpe ratio": (geo - rf_geo) / vol if vol > 0 else np.nan,
        }
    return pd.DataFrame(out).T

stats_df = describe(rets)
stats_df.style.format({
    "Arith. mean": "{:.2%}", "Geo. mean": "{:.2%}", "Volatility": "{:.2%}",
    "Skewness": "{:.2f}", "Excess kurtosis": "{:.2f}", "Sharpe ratio": "{:.2f}",
})

## 6 · Export your results

Run the cell below to download a ZIP with the statistics table (Excel) and the
distribution figure (PNG). In Firefox/Safari you may need to allow pop-ups.

In [ ]:
# Re-create the distribution figure for a clean export
fig_exp, axes = plt.subplots(nrows, ncols, figsize=(13, 3.0 * nrows))
axes = axes.flatten()
for ax, col in zip(axes, rets.columns):
    ax.hist(rets[col], bins=25, color=FHNW["blue"], alpha=0.85, edgecolor="white")
    ax.set_title(col, fontsize=10); ax.set_xlabel("Annual return")
for ax in axes[len(rets.columns):]:
    ax.set_visible(False)
fig_exp.tight_layout()
fig_exp.savefig("distribution.png", dpi=300, bbox_inches="tight")

with pd.ExcelWriter("descriptive_statistics.xlsx") as w:
    stats_df.to_excel(w, sheet_name="Descriptive")
    rets.to_excel(w, sheet_name="Returns")

# Bundle the two output files into one ZIP for download
import zipfile
with zipfile.ZipFile("exercise_01_results.zip", "w") as z:
    z.write("distribution.png")
    z.write("descriptive_statistics.xlsx")

try:
    from google.colab import files
    files.download("exercise_01_results.zip")
except ImportError:
    print("Not running in Colab — files saved to working directory.")